## Prerequisites
1. Create (or reuse) a Kaggle dataset that contains the Sputnik-SR repository **and** the `data/sputnik.db` SQLite file. Attach it to this notebook as an input dataset.
2. Enable the **GPU (T4 x2)** accelerator and make sure internet access is allowed (to install specific TensorFlow wheels).
3. Adjust the hyperparameters in the training command cell if you need different epochs, batch sizes, or evaluation limits.

In [ ]:
# Copy the repository + DB from the attached Kaggle dataset into /kaggle/working
import shutil
from pathlib import Path


INPUT_DATASET = Path("/kaggle/input/sputnik-sr/")
CANDIDATES = sorted(p for p in INPUT_DATASET.iterdir() if (p / "data" / "sputnik.db").exists())
if not CANDIDATES:
    raise FileNotFoundError(
        "Attach a dataset that exposes data/sputnik.db (e.g., /kaggle/input/sputnik-sr/...)."
    )

SOURCE_DIR = CANDIDATES[0]
REPO_DIR = Path("/kaggle/working/sputnik-SR")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(SOURCE_DIR, REPO_DIR)
print("Repository ready at", REPO_DIR)
print("Database located at", REPO_DIR / "data" / "sputnik.db")
# Optional: show git metadata if present
git_info = REPO_DIR / ".git" / "HEAD"
if git_info.exists():
    print("Git HEAD:", git_info.read_text().strip())

In [ ]:
# Install TensorFlow (GPU) and any extra dependencies
!pip install -q --upgrade pip
!pip install -q 'tensorflow==2.15.0' 'tensorflow-io-gcs-filesystem==0.34.0'

In [ ]:
# Verify GPU availability and enable memory growth
import tensorflow as tf


gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise SystemError("No GPU detected. Enable GPU in Kaggle notebook settings.")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:  # noqa: BLE001
        print("Unable to set memory growth on", gpu, exc)
print("Visible GPUs:", gpus)
print("TensorFlow version:", tf.__version__)

In [ ]:
# Setup paths and imports for in-process training
import logging
import sqlite3
import sys
from pathlib import Path


REPO_DIR = Path("/kaggle/working/sputnik-SR")
if str(REPO_DIR) not in sys.path:
    sys.path.append(str(REPO_DIR))

# Configure logging to file AND console
# This ensures logs are saved even if the notebook output buffer overflows
log_file = REPO_DIR / "training.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(log_file, mode="w"), logging.StreamHandler(sys.stdout)],
    force=True,
)
print(f"Logging to {log_file}")

# Import the training function directly
# This avoids creating a subprocess that would compete for GPU memory
from offline_recommender.build_two_towers import build_embeddings


DATABASE = REPO_DIR / "data" / "sputnik.db"
MODELS_DIR = REPO_DIR / "models" / "Two Towers"
CHECKPOINT_PATH = MODELS_DIR / "checkpoints" / "two_towers_kaggle.weights.h5"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# HOTFIX: Patch build_two_towers.py to fix shape mismatch error and reduce verbosity
# This ensures the model output and labels both have shape (batch_size, 1)

file_path = REPO_DIR / "offline_recommender" / "build_two_towers.py"
content = file_path.read_text()

# Fix 1: Remove squeeze layer from model
if "squeeze_score" in content:
    print("Applying fix for model output shape...")
    content = content.replace(
        '    # Expand dimensions for Dense layer: (batch_size,) -> (batch_size, 1)\n    dot_score_expanded = layers.Reshape((1,), name="dot_score_reshape")(dot_score)\n\n    # Map dot product [-1, 1] to rating range using a learned transformation\n    # Add bias and scale to learn the mapping from similarity to rating\n    score_dense = layers.Dense(1, activation=None, use_bias=True, name="score_dense")(\n        dot_score_expanded\n    )\n\n    # Flatten back to (batch_size,) by squeezing the last dimension\n    def squeeze_score(x):\n        return tf.squeeze(x, axis=-1)\n\n    score = layers.Lambda(squeeze_score, output_shape=(1,), name="score")(score_dense)',
        '    # Map dot product [-1, 1] to rating range using a learned transformation\n    # Add bias and scale to learn the mapping from similarity to rating\n    # Output shape: (batch_size, 1)\n    score = layers.Dense(1, activation=None, use_bias=True, name="score")(dot_score)',
    )

# Fix 2: Reshape labels to (N, 1)
if '"labels": np.array(batch_data["labels"], dtype=np.float32),' in content:
    print("Applying fix for labels shape...")
    content = content.replace(
        '"labels": np.array(batch_data["labels"], dtype=np.float32),',
        '"labels": np.array(batch_data["labels"], dtype=np.float32).reshape(-1, 1),',
    )

# Fix 3: Reduce verbosity to avoid IOPub rate limit
# Change verbose=1 to verbose=2 in model.fit calls
if "verbose=1," in content:
    print("Applying fix for verbosity (verbose=1 -> verbose=2)...")
    # We target the specific call in train_model to avoid changing callbacks
    content = content.replace(
        "        callbacks=callbacks,\n        verbose=1,\n        class_weight=class_weight,",
        "        callbacks=callbacks,\n        verbose=2,\n        class_weight=class_weight,",
    )

file_path.write_text(content)
print("Patch applied successfully.")

# Force reload of the module
import sys


if "offline_recommender.build_two_towers" in sys.modules:
    del sys.modules["offline_recommender.build_two_towers"]

In [ ]:
# Run training directly (avoids GPU memory conflicts between notebook and subprocess)
print("Starting Two Towers training...")

# Hyperparameters optimized for T4 GPU / 30GB RAM
with sqlite3.connect(DATABASE) as conn:
    conn.row_factory = sqlite3.Row
    build_embeddings(
        connection=conn,
        embedding_dim=32,
        epochs=20,
        batch_size=512,
        learning_rate=0.001,
        min_user_ratings=10,
        min_release_ratings=3,
        num_negatives=2,
        max_genres=10,
        evaluate_ndcg=True,
        ndcg_holdout_fraction=0.2,
        ndcg_k=9,
        ndcg_max_users=500,
        checkpoint_path=CHECKPOINT_PATH,
        resume_from_checkpoint=CHECKPOINT_PATH,
    )

In [ ]:
# Archive artifacts so they appear under the Kaggle "Output" section for download
import shutil
import zipfile
from pathlib import Path


MODELS_DIR = Path("/kaggle/working/sputnik-SR/models/Two Towers")
OUTPUT_ZIP = Path("/kaggle/working/two_towers_artifacts.zip")
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in MODELS_DIR.rglob("*"):
        zipf.write(path, path.relative_to(MODELS_DIR.parent))
print("Artifacts zipped at", OUTPUT_ZIP)